In [ ]:
using Bessels
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/simulator.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/visualization/plot_settings.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/visualization/plot_utils.jl");

In [ ]:

const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b = 2*ω_a

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b
ω_p=2ω_a-ω_d

ϵ_0 = 0.9#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=12.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 3e-3*κ_2 #from Leon's paper

In [ ]:
t_max=10/κ_b
re_a_0, im_a_0 = 1, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

In [ ]:
ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, system, drive, environment, equation = equation_parity_cos_sin!)

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a, q_CL_a, t_list_CL, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
p1,p2 = plot_mode(ϕ_CL_b[end-5000:end], q_CL_b[end-5000:end], t_list_CL[end-5000:end], ω_b, Z_b)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(b)", ylabel=L"\Im(b)",
title=["Resonator b" "Resonator b RF"],
legend=false)

# Limit cycles

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b = 2*ω_a


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b=π/10.4
κ_a = 0;

In [ ]:
t_max= 30/κ_b
t_min = 0.0

re_a_0, im_a_0 = 2, 2
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

ϵ_p = 0.3
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2
system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

tspan = (t_min, t_max) # Time span for the simulation

#preparation
system_prepared = prepare(system)
drive_prepared = prepare(drive)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
dt_save=2pi/ω_a
t_save = t_max-30/κ_b:dt_save:t_max 
u_0_tot = get_init(system_prepared, environment_prepared)

p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
    
ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = default_solver(u_0_tot, tspan, p, dt; equation=equation_parity_cos_sin!, t_save = t_save)

In [ ]:
ϵ_p_list = range(0.58, .78, 10)#[0.58, 0.6894736842105263, 0.8263157894736842]#

t_max = 30/κ_b
t_min = 0.0
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(30*sqrt(2)*ω_a)
t_min_save = t_max - 1000*dt_save
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

Im_a_list=range(-4,4,10)
Re_a_list=range(-2.5, 2.5 ,25)
Final_all_a = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_a_cat = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b_cat = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_p in ϵ_p_list
    g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_a_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_b_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = (re_i + 1im*im_j)

            system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = equation_parity_cos_sin!)

            a_rot = a.*exp.(1im*ω_a*t_list)*1im*φ_a #90° rotation
            b_rot = b.*exp.(1im*ω_b*t_list)*1im*φ_b #90° rotation

            if abs(maximum(real(a_rot)))<0.2 && abs(minimum(real(a_rot)))>0.15 && maximum(abs.(imag(a_rot)))<0.05
                Final_mode_a_cat[(re_i,im_j)] = a_rot
                Final_mode_b_cat[(re_i,im_j)] = b_rot
            else 
                Final_mode_a[(re_i,im_j)] = a_rot
                Final_mode_b[(re_i,im_j)] = b_rot
            end    
        end
    end
    Final_all_a[ϵ_p] = Final_mode_a
    Final_all_b[ϵ_p] = Final_mode_b
    Final_all_a_cat[ϵ_p] = Final_mode_a_cat
    Final_all_b_cat[ϵ_p] = Final_mode_b_cat
end

In [ ]:

for ϵ_p in ϵ_p_list
    g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b
    Final_mode = Final_all_a[ϵ_p]
    Final_mode_cat = Final_all_a_cat[ϵ_p]
    p0 = plot()

    # Overlay scatter points
    for key in keys(Final_mode_cat)
        z = Final_mode_cat[key]
        scatter!(p0, real.(z), imag.(z), ms = .5, color = RGBA(13/256,76/256,116/256,0), label = "",)
    end

   for key in keys(Final_mode)
        z = Final_mode[key]
        scatter!(p0, real.(z), imag.(z), ms = .5, color = RGBA(185/256,21/256,21/256,0), label = "",)
    end

    # Add labels and title
    xlims!((-0.75, 0.75))
    ylims!((-0.75, 0.75))
    plot!(p0,
          xlabel = L"\mathrm{Re}(a)",
          ylabel = L"\mathrm{Im}(a)",
          title=L"g_2/2\pi="*string(trunc(Int, g2/2pi*1000))*" MHz",
          top_margin=5mm,
          aspect_ratio = 1,)
    display(p0)
    #savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/poster_chicago/poincare_section_09_04_2026_$ϵ_p.png")
end

In [ ]:

for ϵ_p in ϵ_p_list
    g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b
    Final_mode = Final_all_b[ϵ_p]
    Final_mode_cat = Final_all_b_cat[ϵ_p]
    p0 = plot()

    # Overlay scatter points
    for key in keys(Final_mode_cat)
        z = Final_mode_cat[key]
        scatter!(p0, real.(z), imag.(z), ms = .5, color = RGBA(13/256,76/256,116/256,0), label = "",)
    end

   for key in keys(Final_mode)
        z = Final_mode[key]
        scatter!(p0, real.(z), imag.(z), ms = .5, color = RGBA(185/256,21/256,21/256,0), label = "",)
    end

    # Add labels and title
    xlims!((-0.75, 0.75))
    ylims!((-0.75, 0.75))
    plot!(p0,
          xlabel = L"\mathrm{Re}(a)",
          ylabel = L"\mathrm{Im}(a)",
          title=L"g_2/2\pi="*string(trunc(Int, g2/2pi*1000))*" MHz",
          top_margin=5mm,
          aspect_ratio = 1,)
    display(p0)
    #savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/poster_chicago/poincare_section_09_04_2026_$ϵ_p.png")
end